# Upload model artifacts to S3

Pushes the trained model (and the helper artifacts it needs at inference) from
the local `models/` folder up to the same S3 bucket that holds the raw data,
under the `models/` prefix:

```
s3://customer-churn-02/models/churn_model.pkl        <- the joblib model
s3://customer-churn-02/models/feature_columns.pkl
s3://customer-churn-02/models/gender_encoder.pkl
```

Uses `boto3`, which picks up the same AWS credentials already configured for
reading the raw CSV in `data_ingestion.ipynb`.


In [1]:
import boto3

# --- S3 destination -------------------------------------------------------
BUCKET = "customer-churn-02"          # same bucket as the raw training data
MODEL_PREFIX = "models"               # artifacts land under s3://<bucket>/models/

# --- Local artifacts (produced by saving_the_model.ipynb) -----------------
MODELS_DIR = "models"

# The joblib model is the primary artifact. The other two are uploaded with it
# because they are required to reproduce a prediction (feature order + gender
# encoding). Trim this list to just "churn_model.pkl" if you only want the model.
ARTIFACTS = [
    "churn_model.pkl",        # <-- the trained Random Forest (joblib file)
    "feature_columns.pkl",
    "gender_encoder.pkl",
]

In [2]:
import os

s3 = boto3.client("s3")

for name in ARTIFACTS:
    local_path = os.path.join(MODELS_DIR, name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(
            f"{local_path} not found. Run saving_the_model.ipynb first."
        )

    s3_key = f"{MODEL_PREFIX}/{name}"
    s3.upload_file(local_path, BUCKET, s3_key)
    print(f"Uploaded {local_path}  ->  s3://{BUCKET}/{s3_key}")

print("\nDone. Model artifacts are now in S3.")

Uploaded models\churn_model.pkl  ->  s3://customer-churn-02/models/churn_model.pkl
Uploaded models\feature_columns.pkl  ->  s3://customer-churn-02/models/feature_columns.pkl
Uploaded models\gender_encoder.pkl  ->  s3://customer-churn-02/models/gender_encoder.pkl

Done. Model artifacts are now in S3.


### (Optional) Verify the upload

Lists what is now stored under the `models/` prefix in the bucket.


In [3]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{MODEL_PREFIX}/")

for obj in response.get("Contents", []):
    print(f"s3://{BUCKET}/{obj['Key']}   ({obj['Size']} bytes)")

s3://customer-churn-02/models/churn_model.pkl   (33205753 bytes)
s3://customer-churn-02/models/feature_columns.pkl   (167 bytes)
s3://customer-churn-02/models/gender_encoder.pkl   (489 bytes)
